In [0]:
print("Spark version:", spark.version)
print("Catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Schema:", spark.sql("SELECT current_schema()").first()[0])

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/employee_data/"))

In [0]:
%pip install dbldatagen

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("dbldatagen installed successfully")

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F

data_spec = (
    dg.DataGenerator(spark, name="events", rows=1000)
    .withIdOutput()
    .withColumn(
        "category",
        "string",
        values=["A", "B", "C"]
    )
    .withColumn(
        "event_time",
        "timestamp",
        begin="2026-10-13 09:00:00",
        end="2026-10-13 10:00:00",
        interval="1 second"
    )
    .withColumn(
        "value",
        "int",
        minValue=1,
        maxValue=100
    )
)

events_df = data_spec.build()

display(events_df.limit(10))

In [0]:
source_path = "/Volumes/workspace/default/employee_data/lab1/source_events"

(
    events_df.write
    .format("parquet")
    .mode("overwrite")
    .save(source_path)
)

print("Events saved successfully!")

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
saved_events = spark.read.parquet(source_path)

print("Total records:", saved_events.count())

display(saved_events.limit(10))

In [0]:
from pyspark.sql import functions as F

rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numPartitions", 2)
    .load()
)

In [0]:
events_stream = (
    rate_stream
    .withColumn(
        "event_time",
        F.col("timestamp")
    )
    .withColumn(
        "category",
        F.when((F.col("value") % 3) == 0, "A")
         .when((F.col("value") % 3) == 1, "B")
         .otherwise("C")
    )
    .withColumn(
        "event_value",
        (F.col("value") % 100) + 1
    )
    .select(
        "event_time",
        "value",
        "category",
        "event_value"
    )
)

In [0]:
#Apply watermark and windowed aggregation
from pyspark.sql import functions as F

watermarked_agg = (
    events_stream
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        F.window("event_time", "1 minute"),
        F.col("category")
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum("event_value").alias("total_value"),
        F.avg("event_value").alias("avg_value")
    )
)

print("Watermark and aggregation configured successfully")

In [0]:
#Start the stream with a checkpoint
checkpoint_path = (
    "/Volumes/workspace/default/employee_data/"
    "lab1/checkpoints/watermark_lab"
)

# Clear stale checkpoint from previous trigger config
dbutils.fs.rm(checkpoint_path, recurse=True)

query = (
    watermarked_agg.writeStream
    .format("memory")
    .queryName("watermark_aggregation") # hum streaming query execution ke liye naya data hi process ho purna.
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path) # checkpoint save evry processing and executuion history save our query and state (is save stage)
    .trigger(availableNow=True) # Trigger is schdule the time for statring the stream
    #Kan quey ko chlnaha hai aur kitne frequency se update hogi
    .start() #Start the streaming query.
)

print("Streaming query started")
print("Query ID:", query.id)

In [0]:
#Observe the aggregation
print("Active:", query.isActive)
print("Status:", query.status)

display(
    spark.sql("""
        SELECT *
        FROM watermark_aggregation
        ORDER BY window.start DESC
    """)
)

In [0]:
#Kill the stream
#Once the query has run for a while, stop it.
query.stop()

print("Stream stopped")
print("Active:", query.isActive)

In [0]:
#Resume from the same checkpoint
# Clear stale checkpoint from previous trigger config
dbutils.fs.rm(checkpoint_path, recurse=True)

query = (
    watermarked_agg.writeStream
    .format("memory")
    .queryName("watermark_aggregation")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

print("Stream restarted")
print("New query ID:", query.id)

In [0]:
#Verify the streaming query
print("Is query active:", query.isActive)
print("Query status:", query.status)
print("Checkpoint path:", checkpoint_path)

In [0]:
#Stop the streaming query
query.stop()

print("Query stopped successfully")
print("Is query active:", query.isActive)

In [0]:
#Rate Source: Generates continuous synthetic streaming rows.
#dbldatagen: Generates realistic test data for the pipeline.
#Watermark: Controls how long Spark considers late event-time data for stateful processing.
#Windowed aggregation: Calculates event metrics over fixed event-time intervals.
#Checkpoint: Stores progress and state information needed to resume the query.
#Delta sink: Preserves committed output across query restarts.